# **Lecture:** Tracking and video analysis

![](multi-object-tracking.jpg)

### **1. Learning Objectives**


By the end of this lab, you should be able to:

- Explain the difference between **detection** and **tracking**.
- Describe the role of **classical trackers** (KCF, CSRT) and when they are useful.
- Implement a simple **tracking-by-detection** pipeline using PyTorch and OpenCV.
- Run experiments on a **video** and observe how tracking behaves over time.
- Understand how tracking and detection relate to **real systems** like the DeepStream-based pedestrian analytics system described in the master’s thesis proposal (age and gender classification over detected people).


### **2. Detection vs Tracking**

#### 2.1 Object Detection

**Object detection** answers the question: *“What objects are in this frame, and where are they?”*.

- Input: a single image (frame).
- Output: bounding boxes + class labels (e.g., `person`).
- Typical models: Faster R-CNN, YOLO, SSD, etc.
- Cost: relatively **expensive** per frame (deep network forward pass).

#### 2.2 Object Tracking

**Object tracking** answers the question: *“Where did this object move between frames?”*.

- Input: an initial bounding box in frame *t*.
- Output: updated bounding box in frame *t+1, t+2, ...*.
- Trackers assume the object is the **same** and try to follow it over time.
- Cost: usually **cheaper** than running a full detector every frame.

Tracking is useful when:
- You want **smooth trajectories** of objects.
- You want to **reduce computation** by not running detection on every frame.
- You want to maintain **identity** (ID) of each person over time.

#### 2.3 Classical Trackers: KCF and CSRT

OpenCV provides several classical trackers, including:

- **KCF (Kernelized Correlation Filter)**
  - Fast, works well when the object appearance does not change too much.
  - Sensitive to occlusions and large scale changes.

- **CSRT (Discriminative Correlation Filter with Channel and Spatial Reliability)**
  - More accurate than KCF in many cases.
  - Slower than KCF, but still real-time on CPU for moderate resolutions.

These trackers are **not deep learning models**; they are classical computer vision algorithms.

#### 2.4 Tracking-by-Detection

**Tracking-by-detection** combines both ideas:

1. Run a **detector** every N frames (e.g., every 5 or 10 frames).
2. Initialize or update **trackers** for each detected object.
3. Between detection frames, use the trackers to **propagate** object positions.

Advantages:
- More robust than pure tracking (because detection can recover from drift).
- More efficient than running detection on every frame.
- Natural fit for systems that need **IDs over time**, like counting people entering a store or tracking their paths.

In the master’s thesis system, a similar idea is used: detect people, then classify age and gender for each detection, and aggregate statistics over time in a real deployment.

In [1]:
video_path = '/Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4'  # Replace with your video path

In [1]:
from IPython.display import HTML
HTML("""
<video width="640" height="480" controls>
    <source src="/Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4">
</video>
""")

### **KCF (Kernelized Correlation Filter)**
- Type: **Single Object Tracking (SOT)**
- Requires: **Initial bounding box only (first frame)**
- Does NOT use detection after initialization

**Key idea:** Track one object based only on its appearance from the first frame

In [ ]:
import cv2
from ultralytics import YOLO

# Cargar modelo
model = YOLO("yolo26n.pt")

# Leer video
cap = cv2.VideoCapture("race_car.mp4")
ret, frame = cap.read()

if not ret:
    raise Exception("No se pudo leer el video")

# Detectar SOLO en el primer frame
results = model(frame)

car_boxes = []
for box in results[0].boxes:
    if int(box.cls) == 2:
        car_boxes.append(box.xyxy.cpu().numpy())

if len(car_boxes) == 0:
    raise Exception("No se detectó ningún carro en el primer frame")

# Convertir bounding box
box = car_boxes[0].flatten()
x1, y1, x2, y2 = box
bbox = (int(x1), int(y1), int(x2 - x1), int(y2 - y1))

# Inicializar tracker
tracker = cv2.TrackerKCF_create()
tracker.init(frame, bbox)

# LOOP DE VISUALIZACIÓN
while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, "Tracking", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    else:
        cv2.putText(frame, "Lost", (50,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("KCF Tracking", frame)

    # ESC para salir
    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 416x640 2 cars, 43.9ms
Speed: 4.8ms preprocess, 43.9ms inference, 0.1ms postprocess per image at shape (1, 3, 416, 640)


: 

### **CSRT (Discriminative Correlation Filter with Channel and Spatial Reliability)**
- Type: **Single Object Tracking (SOT)**
- Requires: **Initial bounding box only (first frame)**
- Does NOT use detection after initialization

**Key idea:** Improved version of KCF that adapts to scale and appearance changes

In [ ]:
import cv2
from ultralytics import YOLO

# Cargar modelo YOLO
model = YOLO("yolo26n.pt")

# Leer video
cap = cv2.VideoCapture("race_car.mp4")
ret, frame = cap.read()

if not ret:
    raise Exception("No se pudo leer el video")

# Detectar SOLO en el primer frame
results = model(frame)

car_boxes = []
for box in results[0].boxes:
    if int(box.cls) == 2:  # clase "car"
        car_boxes.append(box.xyxy.cpu().numpy())

if len(car_boxes) == 0:
    raise Exception("No se detectó ningún carro en el primer frame")

# Convertir bounding box (xyxy → xywh)
box = car_boxes[0].flatten()
x1, y1, x2, y2 = box
bbox = (int(x1), int(y1), int(x2 - x1), int(y2 - y1))

print("BBox inicial:", bbox)

# 🔥 Inicializar CSRT (más robusto que KCF)
tracker = cv2.TrackerCSRT_create()

# (si falla, usa esta alternativa)
# tracker = cv2.legacy.TrackerCSRT_create()

tracker.init(frame, bbox)

# LOOP de tracking
while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255,0,0), 2)
        cv2.putText(frame, "CSRT Tracking", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)
    else:
        cv2.putText(frame, "Lost", (50,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("CSRT Tracking", frame)

    # ESC para salir
    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 416x640 2 cars, 41.1ms
Speed: 6.3ms preprocess, 41.1ms inference, 1.4ms postprocess per image at shape (1, 3, 416, 640)
BBox inicial: (1309, 412, 145, 101)


: 

### **YOLO + ByteTrack**
- Type: **Multi-Object Tracking (MOT)**
- Requires: **Detection on every frame (YOLO)**
- Uses IoU + confidence score for association

**Key idea:** Fast and efficient tracking-by-detection without deep appearance features

In [ ]:
from ultralytics import YOLO

# Load an official or custom model
model = YOLO("yolo26n.pt")  # Load an official Detect model

#results = model.track(source="/Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4", show=True)  # Tracking with default tracker - BoT-SORT
results = model.track("https://youtu.be/LNwODJXcvt4", show=True, tracker="bytetrack.yaml")  # with ByteTrack

### **Summary**

| Method | Type | Uses Detection Every Frame? | # Objects | Robustness |
|-------|------|----------------------------|----------|-----------|
| KCF | SOT | ❌ No | 1 | Low |
| CSRT | SOT | ❌ No | 1 | Medium |
| YOLO + ByteTrack | MOT | ✅ Yes | Many | High |

**One-Line Intuition**

- KCF → "Follow this object fast"
- CSRT → "Follow this object carefully"
- YOLO + ByteTrack → "Track many objects efficiently"

### **The Tracking Challenge: Motion Trail + Heatmap + Smart Counter**

- Work in **groups of 2 students**

#### Objective

Build a **multi-feature tracking system** using a video of people (the video will be provided by the instructor) that includes:

1. Motion Trail (trajectory visualization)
2. Heatmap of activity
3. Smart Entry Counter

#### Task Overview

You will use an object tracking method (e.g., YOLO + ByteTrack) to:

- Track people across frames
- Extract their positions over time
- Build three different visual/analytical outputs

#### **Task 1: Motion Trail (Trajectory)**

Requirements:
- For each tracked person:
  - Compute the **center point** of the bounding box
  - Draw a **trail (path)** of previous positions
- The trail must:
  - Fade over time (older points disappear gradually)
  - Be updated frame-by-frame

#### **Task 2: Heatmap of Activity**

Requirements:
- Accumulate all detected center points into a spatial grid
- Generate a **heatmap visualization** showing:
  - High-density areas (more traffic)
  - Low-density areas
- Overlay the heatmap on the video or display separately

### **Task 3: Smart Entry Counter**

- Define a **virtual line** (e.g., entrance)
- Count how many people:
  - Cross the line in a specific direction
- Avoid double counting:
  - Each tracked ID should be counted only once

### **Deliverables**

Each group must submit:

1. Working code
2. Video output
3. Short explanation (ONLY ONE of the following per student) of the working logic:
   - Motion Trail
   - Heatmap
   - Counter logic

### **Grading Rubric (10 points total)**

#### Functionality (6 points)

| Component | Points | Criteria |
|----------|--------|---------|
| Motion Trail | 2 pts | Works correctly (center + fading) OR not |
| Heatmap | 2 pts | Correct accumulation and visualization OR not |
| Counter | 2 pts | Correct counting (no duplicates) OR not |

#### Explanation (4 points)

| Criteria | Points |
|--------|--------|
| Clear and correct explanation of chosen component | 4 pts |
| Incorrect or unclear explanation | 0 pts |
- No partial credit

In [2]:
%pip install torch torchvision torchaudio

     ---------------------------------------- 4.0/4.0 MB 15.9 MB/s eta 0:00:00
     ------------------------------------- 328.7/328.7 kB 19.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import cv2
import numpy as np
from collections import defaultdict, deque
from ultralytics import YOLO

# ──────────────────────────────────────────────────────────────
# INICIALIZACIÓN DEL MODELO
# ──────────────────────────────────────────────────────────────
model = YOLO('yolov8s.pt')  # Carga YOLOv8 Small
device = '0' if cv2.cuda.getCudaEnabledDeviceCount() > 0 else 'cpu'  # GPU si está disponible, sino CPU
model.to(device)

# ──────────────────────────────────────────────────────────────
# CONFIGURACIÓN DE VIDEO
# ──────────────────────────────────────────────────────────────
cap    = cv2.VideoCapture('store.mp4')                        # Abre el video de entrada
fps    = int(cap.get(cv2.CAP_PROP_FPS))                       # FPS del video original
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))               # Ancho del frame
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))              # Alto del frame
out    = cv2.VideoWriter('output_tracking.mp4',               # Video de salida
                         cv2.VideoWriter_fourcc(*'mp4v'),      # Codec MP4
                         fps, (width, height))

# ──────────────────────────────────────────────────────────────
# PARÁMETROS DE CONFIGURACIÓN
# ──────────────────────────────────────────────────────────────
line_y     = 390     # Posición Y de la línea virtual de conteo (en píxeles)
TRAIL_LEN  = 40      # Máximo de puntos en el historial de trayectoria
TRAIL_W    = 8       # Grosor máximo de la estela visual
HEAT_DECAY = 0.999   # Decaimiento del heatmap por frame (cercano a 1 = decae lento)
BRIGHTNESS = 20      # Ajuste de brillo (0 = sin cambio)
CONTRAST   = 1.1     # Multiplicador de contraste (1.0 = sin cambio)

# ──────────────────────────────────────────────────────────────
# ESTRUCTURAS DE DATOS
# ──────────────────────────────────────────────────────────────
# Historial de posiciones por persona: {id: deque([(x,y), (x,y), ...])}
# deque con maxlen descarta automáticamente los puntos más antiguos
track_history = defaultdict(lambda: deque(maxlen=TRAIL_LEN))

heatmap_grid            = None   # Matriz del heatmap, se inicializa con el primer frame
entered_ids, exited_ids = set(), set()  # IDs ya contados (evita doble conteo)
entry_count, exit_count = 0, 0          # Contadores totales

# ──────────────────────────────────────────────────────────────
# BUCLE PRINCIPAL DE PROCESAMIENTO
# ──────────────────────────────────────────────────────────────
while cap.isOpened():
    ret, frame = cap.read()   # Lee el siguiente frame
    if not ret:
        break                 # Sin más frames → termina

    # Ajusta brillo (beta) y contraste (alpha) del frame
    frame = cv2.convertScaleAbs(frame, alpha=CONTRAST, beta=BRIGHTNESS)

    # Inicializa el heatmap con ceros en el tamaño del primer frame
    if heatmap_grid is None:
        heatmap_grid = np.zeros((frame.shape[0], frame.shape[1]), dtype=np.float32)

    # Aplica decaimiento: las zonas antiguas se "enfrían" gradualmente
    heatmap_grid *= HEAT_DECAY

    # ──────────────────────────────────────────────────────────
    # DETECCIÓN Y TRACKING CON YOLO + BYTETRACK
    # ──────────────────────────────────────────────────────────
    results = model.track(
        frame,
        persist=True,             # Mantiene el ID entre frames consecutivos
        tracker="bytetrack.yaml", # Algoritmo ByteTrack para asignar IDs
        classes=[0],              # Solo clase 0 = personas (dataset COCO)
        conf=0.5,                 # Confianza mínima del 50%
        imgsz=320,                # Redimensiona internamente para mayor velocidad
        verbose=False             # Sin logs en consola
    )

    annotated = frame.copy()  # Frame donde se dibujan las anotaciones

    # Solo procesa si hay detecciones con ID asignado en este frame
    if results[0].boxes.id is not None:

        boxes_xywh = results[0].boxes.xywh.cpu()              # Bounding boxes: centro (x,y) + tamaño (w,h)
        boxes_xyxy = results[0].boxes.xyxy.cpu().numpy().astype(int)  # Bounding boxes: esquinas (x1,y1,x2,y2)
        track_ids  = results[0].boxes.id.int().cpu().tolist() # IDs únicos asignados por el tracker

        # ──────────────────────────────────────────────────────
        # PROCESAMIENTO POR CADA PERSONA DETECTADA
        # ──────────────────────────────────────────────────────
        for box_xyxy, box_xywh, tid in zip(boxes_xyxy, boxes_xywh, track_ids):
            x1, y1, x2, y2 = box_xyxy          # Esquinas del bounding box
            x, y, w, h     = box_xywh          # Centro y dimensiones
            center          = (int(x), int(y)) # Centro como tupla de enteros

            # Dibuja el bounding box verde y el ID encima
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(annotated, f"ID:{tid}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            # ── Heatmap ────────────────────────────────────────
            # Crea una capa temporal con un círculo en la posición actual
            # y la acumula en el heatmap global (máximo 500 para evitar saturar)
            tmp = np.zeros_like(heatmap_grid)
            cv2.circle(tmp, center, 20, 5, -1)        # Radio=20, valor=5, relleno
            heatmap_grid = np.minimum(heatmap_grid + tmp, 500)

            # ── Estela de trayectoria ──────────────────────────
            track_history[tid].append(center)  # Agrega posición actual al historial
            pts = list(track_history[tid])     # Lista de posiciones recientes

            # Dibuja segmentos con opacidad y grosor crecientes hacia el presente
            for i in range(1, len(pts)):
                p = i / len(pts)   # 0.0 = más antiguo → 1.0 = más reciente
                cv2.line(annotated, pts[i-1], pts[i],
                         (0, int(255*p), 0),       # Verde más brillante cuanto más reciente
                         max(1, int(TRAIL_W*p)))    # Línea más gruesa cuanto más reciente

            # ── CONTEO ENTRADA / SALIDA ────────────────────────
            #
            # En OpenCV el eje Y crece hacia ABAJO:
            #   Y pequeño = parte superior (exterior de la tienda)
            #   Y grande  = parte inferior (interior de la tienda)
            #
            # pts[-2] = posición del frame anterior  →  (x, Y_anterior)
            # pts[-1] = posición del frame actual    →  (x, Y_actual)
            # [1]     = toma solo la coordenada Y de cada tupla
            #
            # ENTRADA: venía de arriba (Y_anterior < line_y)
            #          y ahora está abajo (Y_actual >= line_y)
            #          → cruzó la línea hacia adentro
            #
            # SALIDA:  venía de abajo (Y_anterior > line_y)
            #          y ahora está arriba (Y_actual <= line_y)
            #          → cruzó la línea hacia afuera
            #
            # El set entered_ids/exited_ids garantiza que cada
            # persona se cuente UNA SOLA VEZ aunque siga cerca de la línea
            if len(pts) >= 2:
                if pts[-2][1] < line_y <= pts[-1][1] and tid not in entered_ids:
                    entered_ids.add(tid)   # Marca el ID como contado en entradas
                    entry_count += 1       # Incrementa el contador de entradas

                elif pts[-2][1] > line_y >= pts[-1][1] and tid not in exited_ids:
                    exited_ids.add(tid)    # Marca el ID como contado en salidas
                    exit_count += 1        # Incrementa el contador de salidas

    # ──────────────────────────────────────────────────────────
    # RENDERIZADO DEL HEATMAP
    # ──────────────────────────────────────────────────────────
    blurred    = cv2.GaussianBlur(heatmap_grid, (31, 31), 0)                        # Suaviza el heatmap
    heat_norm  = cv2.normalize(blurred, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)  # Normaliza a 0–255
    color_heat = cv2.applyColorMap(heat_norm, cv2.COLORMAP_JET)                     # Aplica paleta de colores
    mask       = (heat_norm > 2).astype(np.uint8)                                   # Oculta zonas frías
    color_heat = cv2.bitwise_and(color_heat, color_heat, mask=mask)                 # Aplica la máscara
    final      = cv2.addWeighted(annotated, 0.65, color_heat, 0.45, 0)             # Mezcla frame + heatmap

    # ──────────────────────────────────────────────────────────
    # INTERFAZ DE USUARIO
    # ──────────────────────────────────────────────────────────
    cv2.line(final, (0, line_y), (final.shape[1], line_y), (0, 0, 255), 2)          # Línea virtual roja
    cv2.rectangle(final, (10, 10), (300, 80), (0, 0, 0), -1)                        # Fondo del panel
    cv2.putText(final, f"ENTRADAS: {entry_count}", (20, 40),                         # Contador entradas (verde)
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(final, f"SALIDAS:  {exit_count}", (20, 70),                          # Contador salidas (rojo)
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # ──────────────────────────────────────────────────────────
    # ESCRITURA Y VISUALIZACIÓN
    # ──────────────────────────────────────────────────────────
    out.write(final)                        # Guarda el frame en el video de salida
    cv2.imshow("Tracking", final)           # Muestra en pantalla en tiempo real
    if cv2.waitKey(1) & 0xFF == ord('q'):  # Presionar 'q' para salir
        break

# ──────────────────────────────────────────────────────────────
# LIMPIEZA DE RECURSOS
# ──────────────────────────────────────────────────────────────
cap.release()              # Libera la captura de video
out.release()              # Cierra y guarda el video de salida
cv2.destroyAllWindows()    # Cierra todas las ventanas de OpenCV

print(f"Entradas: {entry_count} | Salidas: {exit_count}")  # Resultado final

Entradas: 6 | Salidas: 6


---

<p style="text-align: right; font-size:14px; color:gray;">
<b>Prepared by:</b><br>
Manuel Eugenio Morocho-Cayamcela
</p>